## 1. Project Description

## 2. Data Description and Acquisition

We use three data sources:

**Source 1: Bureau of Transportation Statistics (BTS)** — Our primary dataset. Monthly CSV files downloaded from the [BTS On-Time Performance database](https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ&QO_fu146_anzr=b0-gvzr). Each row is a single flight. Key attributes include carrier, origin/destination airports, scheduled and actual departure/arrival times, delay duration, delay cause, and distance. We pulled all 48 months of 2021-2024. No account required.

**Source 2: NOAA Climate Data Online** — Daily weather summaries pulled via the [NOAA CDO API](https://www.ncdc.noaa.gov/cdo-web/webservices/v2) for the 75 busiest airports. Attributes include max/min temperature, precipitation, snowfall, average wind speed, and max wind gust. Access requires a free API token.

**Source 3: FAA Airport Facilities Data** — Airport metadata from the [FAA ADIP portal](https://adip.faa.gov/agis/public/#/airportSearch/advanced) and [NASR subscription data](https://www.faa.gov/air_traffic/flight_info/aeronav/aero_data/NASR_Subscription/) including latitude/longitude, elevation, and hub classification. Publicly available, no account required.

### 2.1 Imports and Setup

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import requests
import time
import numpy as np
import gc
from scipy import stats

### 2.2 Loading BTS Flight Data

The BTS data was downloaded as monthly CSV files for 2021-2024. We load and concatenate all files into a single dataframe, then convert date fields and derive new time-based features.

In [7]:
files = glob.glob("../../bts/**/*.csv", recursive=True)
## Alison: files = glob.glob("../../data/bts/*.csv")
## Matthew: 
print(f"Found {len(files)} CSV files")

df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)

df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], format='mixed')
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.day_name()
df['MONTH'] = df['FL_DATE'].dt.month
df['YEAR'] = df['FL_DATE'].dt.year

print(f"\nTotal flights: {len(df):,}")
print(f"Date range: {df['FL_DATE'].min()} to {df['FL_DATE'].max()}")
print(f"\nFlights per year:")
print(df['YEAR'].value_counts().sort_index())
print(f"\nColumns: {df.columns.tolist()}")

Found 48 CSV files

Total flights: 26,651,502
Date range: 2021-01-01 00:00:00 to 2024-12-31 00:00:00

Flights per year:
YEAR
2021    5995397
2022    6729125
2023    6847899
2024    7079081
Name: count, dtype: int64

Columns: ['FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'CRS_ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'DAY_OF_WEEK', 'MONTH', 'YEAR']


The BTS delay-cause columns (CARRIER_DELAY, WEATHER_DELAY, NAS_DELAY, SECURITY_DELAY, LATE_AIRCRAFT_DELAY) are only populated when a flight is delayed more than 15 minutes. All other rows are NaN for those fields — this is a structural feature of the dataset, not a data quality issue. We verify this and check cancellation/diversion rates below.

In [10]:
print("BTS Data Quality Check ")
print(f"\nNull counts in key delay columns:")
delay_cols = ['DEP_DELAY', 'ARR_DELAY', 'CARRIER_DELAY', 'WEATHER_DELAY', 
              'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']
print(df[delay_cols].isnull().sum())

print(f"\nCancelled flights: {df['CANCELLED'].sum():,} ({df['CANCELLED'].mean()*100:.2f}%)")
print(f"Diverted flights: {df['DIVERTED'].sum():,} ({df['DIVERTED'].mean()*100:.2f}%)")

print(f"\nDEP_DELAY stats (non-null):")
print(df['DEP_DELAY'].describe())

print(f"\nUnique carriers: {df['OP_UNIQUE_CARRIER'].nunique()}")
print(df['OP_UNIQUE_CARRIER'].value_counts())

BTS Data Quality Check 

Null counts in key delay columns:
DEP_DELAY                455591
ARR_DELAY                532601
CARRIER_DELAY          21427704
WEATHER_DELAY          21427704
NAS_DELAY              21427704
SECURITY_DELAY         21427704
LATE_AIRCRAFT_DELAY    21427704
dtype: int64

Cancelled flights: 468,604.0 (1.76%)
Diverted flights: 63,994.0 (0.24%)

DEP_DELAY stats (non-null):
count    2.619591e+07
mean     1.182002e+01
std      5.318609e+01
min     -1.050000e+02
25%     -5.000000e+00
50%     -2.000000e+00
75%      9.000000e+00
max      4.413000e+03
Name: DEP_DELAY, dtype: float64

Unique carriers: 17
OP_UNIQUE_CARRIER
WN    5229673
DL    3635350
AA    3535381
OO    2906814
UA    2565973
YX    1246327
MQ    1011212
B6     990894
NK     948735
AS     908502
9E     900257
OH     859176
F9     678790
G4     465507
HA     293904
YV     274588
QX     200419
Name: count, dtype: int64


### 2.3 Identifying Top Airports

Rather than pulling weather for every airport in the country, we identify the 75 most common origin and destination airports. These handle the vast majority of domestic flights.

In [13]:
top_origins = df['ORIGIN'].value_counts().head(75).index.tolist()
top_dests = df['DEST'].value_counts().head(75).index.tolist()
top_airports = sorted(list(set(top_origins + top_dests)))
print(f"{len(top_airports)} unique airports to pull weather for")
print(top_airports)

75 unique airports to pull weather for
['ABQ', 'ANC', 'ATL', 'AUS', 'BDL', 'BNA', 'BOI', 'BOS', 'BUF', 'BUR', 'BWI', 'CHS', 'CLE', 'CLT', 'CMH', 'CVG', 'DAL', 'DCA', 'DEN', 'DFW', 'DTW', 'ELP', 'EWR', 'FLL', 'GEG', 'GRR', 'HNL', 'HOU', 'IAD', 'IAH', 'IND', 'JAX', 'JFK', 'KOA', 'LAS', 'LAX', 'LGA', 'MCI', 'MCO', 'MDW', 'MEM', 'MIA', 'MKE', 'MSP', 'MSY', 'OAK', 'OGG', 'OKC', 'OMA', 'ONT', 'ORD', 'ORF', 'PBI', 'PDX', 'PHL', 'PHX', 'PIT', 'RDU', 'RIC', 'RNO', 'RSW', 'SAN', 'SAT', 'SAV', 'SDF', 'SEA', 'SFO', 'SJC', 'SJU', 'SLC', 'SMF', 'SNA', 'STL', 'TPA', 'TUS']


The top 75 origin airports and top 75 destination airports are identical, confirming that the busiest airports dominate both departure and arrival traffic.

### 2.4 Acquiring NOAA Weather Data

We use the NOAA Climate Data Online API to pull daily weather summaries (GHCND dataset) for each of our 75 airports across all four years (2021-2024). The process involves three steps:
1. Define airport coordinates and target weather variables
2. Map each airport to its nearest active NOAA weather station
3. Pull daily observations for 2021-2024

Weather variables collected: max temperature (TMAX), min temperature (TMIN), precipitation (PRCP), snowfall (SNOW), average wind speed (AWND), and max wind gust (WSF2).

**Step 1: Airport coordinates and API setup.**

In [17]:
NOAA_TOKEN = "kuTHNnxyChZZUyUNKiDKqcSmbQpowBOW"
headers = {"token": NOAA_TOKEN}

DATATYPES = ["TMAX", "TMIN", "PRCP", "SNOW", "AWND", "WSF2"]

airport_coords = {
    'ABQ': (35.0402, -106.6090),
    'ANC': (61.1743, -149.9962),
    'ATL': (33.6407, -84.4277),
    'AUS': (30.1975, -97.6664),
    'BDL': (41.9389, -72.6832),
    'BNA': (36.1263, -86.6774),
    'BOI': (43.5644, -116.2228),
    'BOS': (42.3656, -71.0096),
    'BUF': (42.9405, -78.7322),
    'BUR': (34.2007, -118.3585),
    'BWI': (39.1774, -76.6684),
    'CHS': (32.8986, -80.0405),
    'CLE': (41.4058, -81.8539),
    'CLT': (35.2140, -80.9431),
    'CMH': (39.9980, -82.8919),
    'CVG': (39.0488, -84.6678),
    'DAL': (32.8471, -96.8518),
    'DCA': (38.8512, -77.0402),
    'DEN': (39.8561, -104.6737),
    'DFW': (32.8998, -97.0403),
    'DTW': (42.2162, -83.3554),
    'ELP': (31.8072, -106.3776),
    'EWR': (40.6895, -74.1745),
    'FLL': (26.0742, -80.1506),
    'GEG': (47.6199, -117.5338),
    'GRR': (42.8808, -85.5228),
    'HNL': (21.3187, -157.9225),
    'HOU': (29.6454, -95.2789),
    'IAD': (38.9531, -77.4565),
    'IAH': (29.9902, -95.3368),
    'IND': (39.7173, -86.2944),
    'JAX': (30.4941, -81.6879),
    'JFK': (40.6413, -73.7781),
    'KOA': (19.7388, -156.0456),
    'LAS': (36.0840, -115.1537),
    'LAX': (33.9425, -118.4081),
    'LGA': (40.7769, -73.8740),
    'MCO': (28.4312, -81.3081),
    'MCI': (39.2976, -94.7139),
    'MDW': (41.7868, -87.7522),
    'MEM': (35.0424, -89.9767),
    'MIA': (25.7959, -80.2870),
    'MKE': (42.9472, -87.8966),
    'MSP': (44.8848, -93.2223),
    'MSY': (29.9934, -90.2580),
    'OAK': (37.7213, -122.2208),
    'OGG': (20.8986, -156.4305),
    'OKC': (35.3931, -97.6007),
    'OMA': (41.3032, -95.8941),
    'ONT': (34.0560, -117.6012),
    'ORD': (41.9742, -87.9073),
    'ORF': (36.8946, -76.2012),
    'PBI': (26.6832, -80.0956),
    'PDX': (45.5898, -122.5951),
    'PHL': (39.8744, -75.2424),
    'PHX': (33.4373, -112.0078),
    'PIT': (40.4919, -80.2329),
    'RDU': (35.8776, -78.7875),
    'RIC': (37.5052, -77.3197),
    'RNO': (39.4991, -119.7681),
    'RSW': (26.5362, -81.7552),
    'SAN': (32.7338, -117.1933),
    'SAT': (29.5337, -98.4698),
    'SAV': (32.1276, -81.2021),
    'SDF': (38.1744, -85.7360),
    'SEA': (47.4502, -122.3088),
    'SFO': (37.6213, -122.3790),
    'SJC': (37.3626, -121.9290),
    'SJU': (18.4394, -66.0018),
    'SLC': (40.7899, -111.9791),
    'SMF': (38.6954, -121.5908),
    'SNA': (33.6757, -117.8678),
    'STL': (38.7487, -90.3700),
    'TPA': (27.9756, -82.5333),
    'TUS': (32.1161, -110.9410),
}

print(f"{len(airport_coords)} airports ready")

75 airports ready


**Step 2: Map each airport to its nearest NOAA weather station.** We search with an expanding radius, prefer official airport weather stations (USW prefix), and use fallback stations for same-city airports.

In [26]:
airport_to_station = {}
no_station = []

for code, (lat, lon) in airport_coords.items():
    url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/stations"
    
    for radius in [0.01, 0.03, 0.05, 0.075, 0.1, 0.3, 0.5]:
        params = {
            "datasetid": "GHCND",
            "extent": f"{lat-radius},{lon-radius},{lat+radius},{lon+radius}",
            "startdate": "2021-01-01",
            "enddate": "2024-12-31",
            "limit": 25,
            "sortfield": "datacoverage",
            "sortorder": "desc"
        }
        
        resp = requests.get(url, headers=headers, params=params)
        time.sleep(0.3)
        
        if resp.status_code == 200 and resp.json().get("results"):
            results = resp.json()["results"]
            best = None
            for s in results:
                if s["id"].startswith("GHCND:USW"):
                    best = s
                    break
            if not best:
                best = results[0]
            
            airport_to_station[code] = best["id"]
            print(f"{code} -> {best['id']} ({best['name']}) radius={radius}")
            break
    else:
        no_station.append(code)
        print(f"{code} -> NO STATION FOUND")

fallbacks = {
    'HOU': 'IAH',
    'DAL': 'DFW',
    'BUR': 'LAX',
    'ONT': 'LAX',
}

for missing, donor in fallbacks.items():
    if missing in no_station and donor in airport_to_station:
        airport_to_station[missing] = airport_to_station[donor]
        no_station.remove(missing)
        print(f"{missing} -> using {donor}'s station: {airport_to_station[donor]}")

print(f"\nMapped: {len(airport_to_station)} | Failed: {len(no_station)}")
if no_station:
    print(f"Still missing: {no_station}")

ABQ -> GHCND:USW00023050 (ALBUQUERQUE INTERNATIONAL AIRPORT, NM US) radius=0.01
ANC -> GHCND:USC00500275 (ANCHORAGE FORECAST OFFICE, AK US) radius=0.03
ATL -> GHCND:USW00013874 (ATLANTA HARTSFIELD JACKSON INTERNATIONAL AIRPORT, GA US) radius=0.03
AUS -> GHCND:USW00013904 (AUSTIN BERGSTROM INTERNATIONAL AIRPORT, TX US) radius=0.03
BDL -> GHCND:USW00014740 (HARTFORD BRADLEY INTERNATIONAL AIRPORT, CT US) radius=0.01
BNA -> GHCND:USW00013897 (NASHVILLE INTERNATIONAL AIRPORT, TN US) radius=0.03
BOI -> GHCND:USW00024131 (BOISE AIR TERMINAL, ID US) radius=0.03
BOS -> GHCND:USW00014739 (BOSTON LOGAN INTERNATIONAL AIRPORT, MA US) radius=0.01
BUF -> GHCND:USW00014733 (BUFFALO NIAGARA INTERNATIONAL AIRPOR, NY US) radius=0.01
BUR -> GHCND:USW00023152 (BURBANK GLENDALE PASADENA AIRPORT, CA US) radius=0.01
BWI -> GHCND:USW00093721 (BALTIMORE WASHINGTON INTERNATIONAL AIRPORT, MD US) radius=0.03
CHS -> GHCND:USW00013880 (CHARLESTON INTL. AIRPORT, SC US) radius=0.01
CLE -> GHCND:USW00014820 (CLEVELAND 

**Step 3: Pull daily weather observations for all mapped stations across 2021-2024.** This pulls 4 years of data for 75 airports. The API returns a maximum of 1000 records per request, so we paginate through results and handle rate limits automatically.

In [36]:
all_weather = []

years = [
    ("2021-01-01", "2021-12-31"),
    ("2022-01-01", "2022-12-31"),
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31"),
]

for start, end in years:
    print(f"\n{'='*50}")
    print(f"PULLING WEATHER FOR {start[:4]}")
    print(f"{'='*50}")
    
    for airport, station_id in airport_to_station.items():
        print(f"  {airport}...", end=" ")
        
        offset = 1
        retries = 0
        
        while True:
            url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
            params = {
                "datasetid": "GHCND",
                "stationid": station_id,
                "startdate": start,
                "enddate": end,
                "datatypeid": ",".join(DATATYPES),
                "units": "standard",
                "limit": 1000,
                "offset": offset
            }
            
            resp = requests.get(url, headers=headers, params=params)
            
            if resp.status_code == 503:
                retries += 1
                if retries > 3:
                    print(f"giving up at offset {offset}")
                    break
                time.sleep(5)
                continue
            
            if resp.status_code != 200:
                print(f"error {resp.status_code}")
                break
            
            data = resp.json()
            results = data.get("results", [])
            
            if not results:
                break
            
            for r in results:
                r["airport"] = airport
            all_weather.extend(results)
            
            total = data.get("metadata", {}).get("resultset", {}).get("count", 0)
            offset += 1000
            retries = 0
            
            if offset > total:
                break
            
            time.sleep(0.4)
        
        count = sum(1 for r in all_weather if r['airport'] == airport)
        print(f"{count} total")

weather_raw = pd.DataFrame(all_weather)
print(f"\nTotal weather records: {len(weather_raw)}")
print(f"Airports with data: {weather_raw['airport'].nunique()}")


PULLING WEATHER FOR 2021
  ABQ... 2190 total
  ANC... 1457 total
  ATL... 2190 total
  AUS... 2190 total
  BDL... 2190 total
  BNA... 2190 total
  BOI... 2190 total
  BOS... 2190 total
  BUF... 2190 total
  BUR... 1675 total
  BWI... 2190 total
  CHS... 2190 total
  CLE... 2190 total
  CLT... 2190 total
  CMH... 2190 total
  CVG... 2190 total
  DAL... 2190 total
  DCA... 2190 total
  DEN... 2190 total
  DFW... 2190 total
  DTW... 2190 total
  ELP... 2189 total
  EWR... 2190 total
  FLL... 1824 total
  GEG... 2190 total
  GRR... 2190 total
  HNL... 1824 total
  HOU... 1856 total
  IAD... 2190 total
  IAH... 2190 total
  IND... 2190 total
  JAX... 2190 total
  JFK... 2190 total
  KOA... 1820 total
  LAS... 2190 total
  LAX... 1825 total
  LGA... 2190 total
  MCO... 2190 total
  MCI... 2190 total
  MDW... 1822 total
  MEM... 2190 total
  MIA... 1825 total
  MKE... 2190 total
  MSP... 2190 total
  MSY... 1823 total
  OAK... 1824 total
  OGG... 1825 total
  OKC... 2190 total
  OMA... 2190 

In [16]:
weather_raw.to_csv("../../weather_raw.csv", index=False)
print(f"Saved {len(weather_raw)} raw weather records to weather_raw.csv"

Saved 629,166 raw weather records to weather_raw.csv


### 2.5 FAA Airport Facilities Data

We load the FAA's NASR subscription airport data and filter to our 75 airports.

In [20]:
faa_raw = pd.read_csv("../../faa/APT_BASE.csv", encoding='latin-1', low_memory=False)

our_airports = sorted(list(airport_coords.keys()))

faa = faa_raw[faa_raw['ARPT_ID'].isin(our_airports)].copy()
print(f"Matched {len(faa)} of {len(our_airports)} airports")

keep_cols = {
    'ARPT_ID': 'airport',
    'ARPT_NAME': 'airport_name',
    'CITY': 'city',
    'STATE_CODE': 'state',
    'LAT_DECIMAL': 'latitude',
    'LONG_DECIMAL': 'longitude',
    'ELEV': 'elevation_ft',
}
available = {k: v for k, v in keep_cols.items() if k in faa.columns}
faa = faa[list(available.keys())].rename(columns=available)

faa['elevation_ft'] = pd.to_numeric(faa['elevation_ft'], errors='coerce')
faa['latitude'] = pd.to_numeric(faa['latitude'], errors='coerce')
faa['longitude'] = pd.to_numeric(faa['longitude'], errors='coerce')

print(f"\nFAA columns: {faa.columns.tolist()}")
print(f"Shape: {faa.shape}")
faa.head(10)

Matched 75 of 75 airports

FAA columns: ['airport', 'airport_name', 'city', 'state', 'latitude', 'longitude', 'elevation_ft']
Shape: (75, 7)


,airport,airport_name,city,state,latitude,longitude,elevation_ft
518,PHX,PHOENIX SKY HARBOR INTL,PHOENIX,AZ,33.434278,-112.011583,1134.8
611,TUS,TUCSON INTL,TUCSON,AZ,32.117070,-110.941463,2643.0
1058,BUR,BOB HOPE,BURBANK,CA,34.200694,-118.358667,778.0
1419,LAX,LOS ANGELES INTL,LOS ANGELES,CA,33.942496,-118.408049,127.8
1500,OAK,OAKLAND SAN FRANCISCO BAY,OAKLAND,CA,37.721261,-122.221151,9.0
1511,ONT,ONTARIO INTL,ONTARIO,CA,34.056014,-117.601187,944.1
1616,SMF,SACRAMENTO INTL,SACRAMENTO,CA,38.695444,-121.590778,26.9
1648,SAN,SAN DIEGO INTL,SAN DIEGO,CA,32.733563,-117.189663,16.8
1662,SFO,SAN FRANCISCO INTL,SAN FRANCISCO,CA,37.618806,-122.375417,13.1
1670,SJC,NORMAN Y MINETA SAN JOSE INTL,SAN JOSE,CA,37.362995,-121.928621,62.2


## 3. Data Cleaning

### 3.1 Cleaning NOAA Weather Data

The raw NOAA data comes in long format, with one row per measurement per day. We first load the cached raw data, then reshape it into wide format so each row is one airport-day with columns for each weather variable.

In [23]:
if os.path.exists("../../weather_raw.csv"):
    weather_raw = pd.read_csv("../../weather_raw.csv")
    print(f"Loaded cached weather data: {len(weather_raw)} records")
else:
    print("No cached weather file found — run Step 2.4.3 to pull from NOAA API")

Loaded cached weather data: 629166 records


In [25]:
weather_raw['date'] = pd.to_datetime(weather_raw['date'])
weather_raw = weather_raw[['airport', 'date', 'datatype', 'value']]
weather_raw = weather_raw.drop_duplicates()

weather = weather_raw.pivot_table(
    index=['airport', 'date'],
    columns='datatype',
    values='value',
    aggfunc='first'
).reset_index()

weather.columns.name = None

rename_map = {
    'TMAX': 'temp_max', 'TMIN': 'temp_min',
    'PRCP': 'precipitation', 'SNOW': 'snowfall',
    'AWND': 'avg_wind', 'WSF2': 'max_wind_gust'
}
weather = weather.rename(columns={k: v for k, v in rename_map.items() if k in weather.columns})

print(f"Shape: {weather.shape}")
print(f"Date range: {weather['date'].min()} to {weather['date'].max()}")
print(f"\nMissing values before cleaning:")
print(weather.isnull().sum())

Shape: (109539, 8)
Date range: 2021-01-01 00:00:00 to 2024-12-31 00:00:00

Missing values before cleaning:
airport              0
date                 0
avg_wind          1805
precipitation       72
snowfall         24238
temp_max            69
temp_min            83
max_wind_gust     1801
dtype: int64


**Step 1: Base imputation.** We fill snowfall NaNs with 0 (no snowfall recorded), fill missing wind gusts with same-day average wind speed, and interpolate short temperature gaps within each airport.

In [28]:
# Snowfall: NaN means no snowfall recorded
if 'snowfall' in weather.columns:
    weather['snowfall'] = weather['snowfall'].fillna(0)

# Max wind gust: fill with same-day avg wind as first pass
if 'max_wind_gust' in weather.columns and 'avg_wind' in weather.columns:
    weather['max_wind_gust'] = weather['max_wind_gust'].fillna(weather['avg_wind'])

# Temperature: interpolate short gaps within each airport
for col in ['temp_max', 'temp_min']:
    if col in weather.columns:
        weather[col] = weather.groupby('airport')[col].transform(
            lambda x: x.interpolate(limit=3)
        )

print("Step 1 complete")
print(weather[['snowfall', 'max_wind_gust', 'temp_max', 'temp_min']].isnull().sum())

Step 1 complete
snowfall            0
max_wind_gust    1800
temp_max           50
temp_min           51
dtype: int64


**Step 2: Per-airport median fill.** For any remaining NaNs in wind, precipitation, and temperature columns, we fill with that airport's own median. This preserves geographic differences — wind in Anchorage should not be imputed with data from Phoenix. A global median fallback catches edge cases where an entire airport column is missing.

In [31]:
for col in ['avg_wind', 'max_wind_gust', 'precipitation', 'temp_max', 'temp_min']:
    if col in weather.columns:
        weather[col] = weather.groupby('airport')[col].transform(
            lambda x: x.fillna(x.median())
        )

# Global median fallback for any airport with entirely missing columns
weather['avg_wind'] = weather['avg_wind'].fillna(weather['avg_wind'].median())
weather['max_wind_gust'] = weather['max_wind_gust'].fillna(weather['max_wind_gust'].median())

print("Missing values after imputation:")
print(weather[['avg_wind', 'precipitation', 'temp_max', 'temp_min', 'max_wind_gust', 'snowfall']].isnull().sum())

Missing values after imputation:
avg_wind         0
precipitation    0
temp_max         0
temp_min         0
max_wind_gust    0
snowfall         0
dtype: int64


**Step 3: Derived features.** With all raw columns fully imputed, we compute temperature range, binary severity indicators for precipitation (>1 inch), snowfall (>2 inches), and wind speed (>15 mph), and a composite weather severity score averaging the three indicators. Computing these after imputation ensures no NaN values are silently treated as non-severe.

In [34]:
if 'temp_max' in weather.columns and 'temp_min' in weather.columns:
    weather['temp_range'] = weather['temp_max'] - weather['temp_min']

severity_cols = []
if 'precipitation' in weather.columns:
    weather['precip_severe'] = (weather['precipitation'] > 1.0).astype(int)
    severity_cols.append('precip_severe')
if 'snowfall' in weather.columns:
    weather['snow_severe'] = (weather['snowfall'] > 2.0).astype(int)
    severity_cols.append('snow_severe')
if 'avg_wind' in weather.columns:
    weather['wind_severe'] = (weather['avg_wind'] > 15).astype(int)
    severity_cols.append('wind_severe')
if severity_cols:
    weather['weather_severity'] = weather[severity_cols].sum(axis=1) / len(severity_cols)

print(f"\nShape: {weather.shape}")
print(f"\nMissing values after cleaning:")
print(weather.isnull().sum())
print(f"\nColumns: {weather.columns.tolist()}")


Shape: (109539, 13)

Missing values after cleaning:
airport             0
date                0
avg_wind            0
precipitation       0
snowfall            0
temp_max            0
temp_min            0
max_wind_gust       0
temp_range          0
precip_severe       0
snow_severe         0
wind_severe         0
weather_severity    0
dtype: int64

Columns: ['airport', 'date', 'avg_wind', 'precipitation', 'snowfall', 'temp_max', 'temp_min', 'max_wind_gust', 'temp_range', 'precip_severe', 'snow_severe', 'wind_severe', 'weather_severity']


### 3.2 Cleaning FAA Airport Data

We add hub size classifications from the FAA's ADIP portal and encode them as ordinal values for use in regression models.

In [37]:
hub_sizes = {
    'ATL':'L','AUS':'L','BNA':'L','BOS':'L','BWI':'L','CLT':'L','DCA':'L',
    'DEN':'L','DFW':'L','DTW':'L','EWR':'L','FLL':'L','HNL':'L','IAD':'L',
    'IAH':'L','JFK':'L','LAS':'L','LAX':'L','LGA':'L','MCO':'L','MDW':'L',
    'MIA':'L','MSP':'L','ORD':'L','PHL':'L','PHX':'L','SAN':'L','SEA':'L',
    'SFO':'L','SLC':'L','TPA':'L',
    'ABQ':'M','ANC':'M','BDL':'M','BOI':'M','BUF':'M','BUR':'M','CHS':'M',
    'CLE':'M','CMH':'M','CVG':'M','DAL':'M','HOU':'M','IND':'M','JAX':'M',
    'MCI':'M','MKE':'M','MSY':'M','OAK':'M','OGG':'M','OMA':'M','ONT':'M',
    'PBI':'M','PDX':'M','PIT':'M','RDU':'M','RSW':'M','SAT':'M','SJC':'M',
    'SJU':'M','SMF':'M','SNA':'M','STL':'M',
    'ELP':'S','GEG':'S','GRR':'S','KOA':'S','MEM':'S','OKC':'S',
    'ORF':'S','RIC':'S','RNO':'S','SAV':'S','SDF':'S','TUS':'S',
}

faa['hub_size'] = faa['airport'].map(hub_sizes)
hub_map = {'L': 4, 'M': 3, 'S': 2, 'N': 1}
faa['hub_ordinal'] = faa['hub_size'].map(hub_map)

print(f"Hub distribution:")
print(faa['hub_size'].value_counts())
print(f"\nMissing values:")
print(faa.isnull().sum())
faa.head(10)

Hub distribution:
hub_size
M    32
L    31
S    12
Name: count, dtype: int64

Missing values:
airport         0
airport_name    0
city            0
state           0
latitude        0
longitude       0
elevation_ft    0
hub_size        0
hub_ordinal     0
dtype: int64


,airport,airport_name,city,state,latitude,longitude,elevation_ft,hub_size,hub_ordinal
518,PHX,PHOENIX SKY HARBOR INTL,PHOENIX,AZ,33.434278,-112.011583,1134.8,L,4
611,TUS,TUCSON INTL,TUCSON,AZ,32.117070,-110.941463,2643.0,S,2
1058,BUR,BOB HOPE,BURBANK,CA,34.200694,-118.358667,778.0,M,3
1419,LAX,LOS ANGELES INTL,LOS ANGELES,CA,33.942496,-118.408049,127.8,L,4
1500,OAK,OAKLAND SAN FRANCISCO BAY,OAKLAND,CA,37.721261,-122.221151,9.0,M,3
1511,ONT,ONTARIO INTL,ONTARIO,CA,34.056014,-117.601187,944.1,M,3
1616,SMF,SACRAMENTO INTL,SACRAMENTO,CA,38.695444,-121.590778,26.9,M,3
1648,SAN,SAN DIEGO INTL,SAN DIEGO,CA,32.733563,-117.189663,16.8,L,4
1662,SFO,SAN FRANCISCO INTL,SAN FRANCISCO,CA,37.618806,-122.375417,13.1,L,4
1670,SJC,NORMAN Y MINETA SAN JOSE INTL,SAN JOSE,CA,37.362995,-121.928621,62.2,M,3


### 3.3 Merging Datasets

We merge the BTS flight data with NOAA weather data on airport code and date, then merge with FAA airport metadata on origin airport code. This allows us to analyze how weather conditions and airport characteristics relate to delay outcomes. We use left joins so that all 26.6 million flights are preserved — flights at airports outside our top 75 or on dates with missing weather data will have NaN in the weather columns.

In [40]:
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

# merge weather
df_merged = df.merge(
    weather,
    left_on=['ORIGIN', 'FL_DATE'],
    right_on=['airport', 'date'],
    how='left'
)
df_merged = df_merged.drop(columns=['airport', 'date'])

weather_matched = df_merged['temp_max'].notna().sum()
print(f"Flights with weather: {weather_matched:,}/{len(df_merged):,} ({weather_matched/len(df_merged)*100:.1f}%)")

Flights with weather: 23,035,422/26,651,502 (86.4%)


In [42]:
try:
    del df
except:
    pass
try:
    del weather
except:
    pass
try:
    del weather_raw
except:
    pass
try:
    del faa_raw
except:
    pass

gc.collect()
print(f"df_merged: {df_merged.shape}")

df_merged: (26651502, 47)


In [44]:
# merge FAA
df_merged = df_merged.merge(
    faa,
    left_on='ORIGIN',
    right_on='airport',
    how='left',
    suffixes=('', '_faa')
)
if 'airport' in df_merged.columns:
    df_merged = df_merged.drop(columns=['airport'])

faa_matched = df_merged['hub_size'].notna().sum()
print(f"Flights with FAA: {faa_matched:,}/{len(df_merged):,} ({faa_matched/len(df_merged)*100:.1f}%)")

print(f"\nFinal dataset: {df_merged.shape}")
print(f"Date range: {df_merged['FL_DATE'].min()} to {df_merged['FL_DATE'].max()}")
print(f"Columns: {df_merged.columns.tolist()}")

Flights with FAA: 23,038,610/26,651,502 (86.4%)

Final dataset: (26651502, 55)
Date range: 2021-01-01 00:00:00 to 2024-12-31 00:00:00
Columns: ['FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST_AIRPORT_ID', 'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'CRS_ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'DAY_OF_WEEK', 'MONTH', 'YEAR', 'avg_wind', 'precipitation', 'snowfall', 'temp_max', 'temp_min', 'max_wind_gust', 'temp_range', 'precip_severe', 'snow_severe', 'wind_severe', 'weather_severity', 'airport_name', 'city', 'state', 'latitude', 'longitude', 'elevation_ft', 'hub_size', 'hub_ordinal']


In [46]:
df_merged.to_csv("../../merged_full.csv", index=False)
print(f"Saved to merged_full.csv")

Saved to merged_full.csv


In [48]:
print(f"Merged Dataset Summary")
print(f"Total flights: {len(df_merged):,}")
print(f"With weather data: {df_merged['temp_max'].notna().sum():,} ({df_merged['temp_max'].notna().mean()*100:.1f}%)")
print(f"With FAA data: {df_merged['hub_size'].notna().sum():,} ({df_merged['hub_size'].notna().mean()*100:.1f}%)")
print(f"Cancelled: {df_merged['CANCELLED'].sum():,}")
print(f"DEP_DELAY null: {df_merged['DEP_DELAY'].isnull().sum():,}")

Merged Dataset Summary
Total flights: 26,651,502
With weather data: 23,035,422 (86.4%)
With FAA data: 23,038,610 (86.4%)
Cancelled: 468,604.0
DEP_DELAY null: 455,591

For weather analysis, filter to rows with weather data:
  df_weather = df_merged[df_merged['temp_max'].notna()]

For delay analysis, exclude cancelled flights:
  df_delays = df_merged[df_merged['CANCELLED'] == 0]


## 4. Ethical Data Concerns

## 5. Methods Used

## 6. Preliminary Results

### Setup

We create filtered subsets for different analysis needs. Non-cancelled flights for delay analysis (cancelled flights have no delay duration). Flights with weather data for weather-specific questions.

In [ ]:
import numpy as np
from scipy import stats

df_merged = pd.read_csv("../../merged_full.csv", low_memory=False)
df_merged['FL_DATE'] = pd.to_datetime(df_merged['FL_DATE'])

# Non-cancelled flights for delay analysis
df_delays = df_merged[df_merged['CANCELLED'] == 0].copy()

# Flights with weather data
df_weather = df_delays[df_delays['temp_max'].notna()].copy()

print(f"All flights: {len(df_merged):,}")
print(f"Non-cancelled (for delay analysis): {len(df_delays):,}")
print(f"With weather (for weather analysis): {len(df_weather):,}")

### Q1: Which airlines have the best/worst on-time performance?

We measure on-time performance two ways: average departure delay in minutes, and the percentage of flights delayed more than 15 minutes (DEP_DEL15). Both matter — average delay captures severity, while delay rate captures frequency.

In [54]:
carrier_stats = df_delays.groupby('OP_UNIQUE_CARRIER').agg(
    flights=('DEP_DELAY', 'count'),
    avg_delay=('DEP_DELAY', 'mean'),
    median_delay=('DEP_DELAY', 'median'),
    std_delay=('DEP_DELAY', 'std'),
    delay_rate=('DEP_DEL15', 'mean'),
    avg_arr_delay=('ARR_DELAY', 'mean')
).round(2)

carrier_stats = carrier_stats.sort_values('avg_delay', ascending=False)
print("Carrier On-Time Performance (sorted by avg departure delay):")
print(carrier_stats.to_string())

NameError: name 'df_delays' is not defined

In [ ]:

#Q3 - Which airports cause the biggest delays across networks? (which are the biggest bottlenecks) - group by origin airport, calculate average departure delay

#Q4 - What are common causes of delays? - Sum each delay by column across all flights

#Q5 - Are certain routes worse than others? - Create a route column?

#Q6 - Do delays vary by season? - categorize seasons into certain months and compare average delays

#Q7 - Does weather significantly predict delay length, or is it less impactful than people assume? - calculate total weather driven delays

#Q8 - Are delays getting better or worse over time? - group by months and year and calculate average over time

#Q9 - Is there a relationship between flight distance and delay likelihood? - linear regression 


## Interactive Map Prototype